# W01 Wed — F1 Data Ecosystem
## IIT414W · Unit I · Week 1 · Wed March 4, 2026

**Learning goals for this session:**
- Navigate FastF1 for race results, lap data, telemetry, and weather.
- Query the Jolpica API (Ergast successor) for historical results and standings.
- Understand the structure of F1 data: temporal, hierarchical, multi-modal.
- Identify at least 3 potential ML prediction targets from F1 data and articulate why each is interesting.
- Write your first structured PROMPTS.md entry documenting an AI interaction.

**Before you start:** Re-run W01_Mon and confirm FastF1 cache setup works on your machine.

**Estimated time:** 75 min theory + 55 min guided practice + 20 min reflection = 150 min

In [ ]:
# ── Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')


In [ ]:
# ── Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'requests': 'requests',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import numpy as np               # Numeric utilities
import pandas as pd              # Tables and summaries
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Visualization helpers
import requests                  # Ergast/OpenF1 API calls
import fastf1                    # FastF1 session access

print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

## 0. The Hook — What Can F1 Data Actually Tell Us?

**Before any theory**, let's see what three lines of code can reveal.

We'll load the 2024 Monaco GP qualifying session and plot the lap times for the top 5 qualifiers. Look at the chart and ask yourself:

> *"If you were an engineer at McLaren and you saw this data — what question would you ask? What would you want to predict?"*

In [ ]:
# ── The Hook: Monaco 2024 Qualifying ──────────────────────────────────────────
# Three lines to load a session, then we visualise qualifying gaps for the top 5.
import os, fastf1, fastf1.plotting, pandas as pd, matplotlib.pyplot as plt

cache_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'fastf1_cache'))
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

# Load 2024 Monaco Qualifying
quali = fastf1.get_session(2024, 'Monaco', 'Q')
quali.load(laps=True, telemetry=False, weather=False, messages=False)

# ── Use the OFFICIAL qualifying results (not just fastest laps) ───────────────
# quali.results contains the final classification with Position, Q1/Q2/Q3 times.
# This is more robust than inferring position from all-session fastest laps.
top5 = quali.results.sort_values('Position').head(5).copy()

# Helper: format timedelta → "M:SS.mmm"
def fmt_lap(td):
    if pd.isna(td):
        return ''
    total = td.total_seconds()
    mins, secs = divmod(total, 60)
    return f'{int(mins)}:{secs:06.3f}'

# Create readable columns
top5['Q1_fmt'] = top5['Q1'].apply(fmt_lap)
top5['Q2_fmt'] = top5['Q2'].apply(fmt_lap)
top5['Q3_fmt'] = top5['Q3'].apply(fmt_lap)
top5['Q3_sec'] = top5['Q3'].dt.total_seconds()
pole_time = top5['Q3_sec'].iloc[0]
top5['Gap'] = top5['Q3_sec'] - pole_time
top5['Gap_str'] = top5['Gap'].apply(lambda g: 'POLE' if g == 0 else f'+{g:.3f}s')

# Build a "Driver — Team" label so students can see the driver–team relationship
top5['Label'] = top5['Abbreviation'] + ' — ' + top5['TeamName']

# Show the DataFrame — point out columns: Driver, Team, Q1–Q3, Gap
print("Top 5 Qualifying Results — 2024 Monaco GP\n")
print(top5[['Abbreviation', 'TeamName', 'Q1_fmt', 'Q2_fmt', 'Q3_fmt', 'Gap_str']]
      .rename(columns={'Abbreviation': 'Driver', 'TeamName': 'Team',
                        'Q1_fmt': 'Q1', 'Q2_fmt': 'Q2',
                        'Q3_fmt': 'Q3', 'Gap_str': 'Gap'})
      .to_string(index=False))

# ── Plot: Gap to pole ────────────────────────────────────────────────────────
# Instead of raw lap times (all ~70s, visually identical), we plot the GAP TO
# POLE — this is what broadcasters and engineers actually look at.
# Use official team colours via fastf1.plotting for immediate visual recognition.
team_colors = [
    fastf1.plotting.get_team_color(team, session=quali)
    for team in top5['TeamName']
]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(
    top5['Label'],
    top5['Gap'],
    color=team_colors,
    edgecolor='white',
)
# Annotate each bar with the exact gap
for bar, gap in zip(bars, top5['Gap']):
    label = 'POLE' if gap == 0 else f'+{gap:.3f}s'
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            label, va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Gap to Pole (seconds)')
ax.set_title('2024 Monaco GP Qualifying — Gap to Pole')
ax.invert_yaxis()  # P1 at top
ax.set_xlim(0, top5['Gap'].max() * 1.4)  # breathing room for labels
sns.despine(left=True)
plt.tight_layout()
plt.show()

## 1. The F1 Data Landscape

**The idea.** F1 data is hierarchical and temporal, with different useful granularities for different ML tasks.

**In F1 terms.** Session-level outcomes and telemetry streams answer different engineering questions.

> **Why it matters:** Choosing the wrong granularity can break features and create leakage.


### Granularity × Data Type × Source

| Granularity | Examples | Frequency | Source |
|---|---|---|---|
| Season | Standings, constructor pts | 1/year | Jolpica |
| Race | Results, weather, DNFs | ~22/year | Jolpica, FastF1 |
| Lap | Lap times, pit stops, tire compound | ~60/driver/race | FastF1 |
| Sector | Sector times, mini-sectors | 3/lap | FastF1 |
| Telemetry | Speed, throttle, brake, RPM, DRS | ~300 Hz | FastF1 |

> **Key point:** *This is temporal, hierarchical, multi-modal data. It's messy in exactly the ways that make ML interesting — and exactly the ways that create leakage if you're not careful.*

### Public vs. Proprietary Data

What we have access to (FastF1, Jolpica, OpenF1) vs. what teams have (wind tunnel data, tire degradation models, CFD simulations, private telemetry channels).

> *"We are building models with publicly available data. Real teams have much more. The skills transfer — the data won't."*

### API Landscape

| API | Type | Best For | Notes |
|---|---|---|---|
| **FastF1** | Python library | Lap-level data, telemetry, weather | Wraps F1 timing data. Needs local cache. Primary tool for this course. |
| **Jolpica** | REST API | Historical results, standings, circuits | Successor to Ergast (retired end of 2024). Long-horizon archive. |
| **OpenF1** | REST API | Real-time + historical data | Free tier. Good complement. Optional for this course. |

> *"For this course, FastF1 is your primary tool. Jolpica is for historical context. OpenF1 is optional but interesting."*

### 🤔 Before you code: make a decision

Answer these questions in this cell before running any code below.

1. **What question are we trying to answer?**
   > What granularity is appropriate for a pre-race Top-10 prediction and why?

2. **What does success look like? How will we know if the result is good?**
   > [Student fills this in]

3. **What could go wrong? Name one failure mode.**
   > [Student fills this in]


In [ ]:
landscape = pd.DataFrame(
    {
        'granularity': ['session', 'lap', 'sector', 'telemetry'],
        'example': ['winner', 'lap time', 'sector split', 'speed/throttle/brake'],
        'typical_use': ['outcome analysis', 'pace modeling', 'corner diagnostics', 'driving style analysis'],
    }
)
print(landscape.to_string(index=False))


In [ ]:
# ── YOUR TURN (5 min) ─────────────────────────────────────────────
# The `landscape` DataFrame above has 4 rows: session, lap, sector, telemetry.
# It is MISSING an important F1 data type: pit-stop event logs.
#
# Do the following — ALL THREE STEPS:
#
# Step 1. Create a NEW one-row DataFrame called `new_row` with these EXACT columns:
#           granularity  → 'event'
#           example      → 'pit-stop duration, in-lap, out-lap'
#           typical_use  → <your answer: a concrete ML use case, e.g. 'pit strategy optimization'>
#
# Step 2. Append it to `landscape` using pd.concat():
#           landscape = pd.concat([landscape, new_row], ignore_index=True)
#
# Step 3. Print the updated DataFrame:
#           print(landscape.to_string(index=False))
#
# ✅ Expected output: A 5-row table printed to the screen.
# ❌ Common mistake: Forgetting ignore_index=True (you'll get duplicate index 0).

# TODO: Write your code here
raise NotImplementedError('Replace this line with your implementation.')

### 📝 Reflection

Answer the following in this cell (2-4 sentences each):

1. **What did you learn from this exercise that surprised you?**
2. **Where could this technique fail or mislead you?**
3. **How would you explain this to a non-technical teammate?**


## 2. FastF1 Deep Dive

**The idea.** FastF1 provides structured access to race results, laps, telemetry, and weather.

**In F1 terms.** This mirrors race engineering workflows where multiple channels are fused for decisions.

> **Why it matters:** Feature engineering quality depends on understanding each table before modeling.


### 🤔 Before you code: make a decision

Answer these questions in this cell before running any code below.

1. **What question are we trying to answer?**
   > Which FastF1 tables are available pre-race vs. only after the session finishes?

2. **What does success look like? How will we know if the result is good?**
   > [Student fills this in]

3. **What could go wrong? Name one failure mode.**
   > [Student fills this in]


In [ ]:
# FastF1 cache — auto-create if missing
# The cache stores downloaded session data locally so you don't re-download
# it every time you run the notebook (sessions can be hundreds of MB)
import os
cache_path = os.path.join(os.path.dirname(os.getcwd()), '..', 'data', 'fastf1_cache')
cache_path = os.path.abspath(cache_path)
os.makedirs(cache_path, exist_ok=True)          # create folder if it doesn't exist yet
fastf1.Cache.enable_cache(cache_path)           # tell FastF1 where to store cached files
print(f'FastF1 cache enabled: {cache_path}')

# ── Load a session ────────────────────────────────────────────────────────────
# get_session(year, event, session_type)
#   year         : F1 season (e.g. 2024)
#   event        : circuit name, round number, or GP name (e.g. 'Monaco')
#   session_type : 'R' = Race, 'Q' = Qualifying, 'FP1/FP2/FP3' = Practice
session = fastf1.get_session(2024, 'Monaco', 'R')

# session.load() fetches all data for the session.
# Each flag controls a separate data channel — set to False what you don't need
# to speed up loading:
#   laps=True      : per-lap timings, tyre info, sector splits for every driver
#   telemetry=True : high-frequency car sensor data (speed, throttle, brake, etc.)
#   weather=True   : track/air temperature, wind, humidity sampled every ~60 s
#   messages=False : race control messages (flags, penalties) — not needed here
session.load(laps=True, telemetry=True, weather=True, messages=False)

# ── Extract the main data tables ──────────────────────────────────────────────
# session.results    : one row per driver — final classification, points, grid pos
# session.laps       : one row per lap per driver — times, compounds, positions
# session.weather_data: time-series of weather measurements during the session

results_df = session.results.copy()      # copy() so we don't accidentally modify the live object
laps_df    = session.laps.copy()
weather_df = session.weather_data.copy()

# ── Fastest lap telemetry ─────────────────────────────────────────────────────
# pick_fastest() returns the single Lap object with the shortest valid LapTime
# across ALL drivers in the session
fastest = session.laps.pick_fastest()

# get_car_data() returns the raw telemetry stream for that lap:
#   RPM, Speed (km/h), nGear, Throttle (%), Brake (bool), DRS status
# add_distance() integrates speed over time to compute a 'Distance' column (m)
# — useful for aligning data spatially around the circuit rather than by time
car_df = fastest.get_car_data().add_distance()

# ── Inspect each DataFrame ────────────────────────────────────────────────────
# For each table we print:
#   shape   : (rows, columns) — gives a quick size sanity check
#   missing : total NaN count — flags data quality issues before modeling
#   dtypes  : column types — important for choosing correct transformations
for name, df in [('results', results_df), ('laps', laps_df), ('weather', weather_df), ('car_data', car_df)]:
    print(f"\n{'=' * 60}")
    print(f"  {name.upper()}  |  shape={df.shape}  |  missing={int(df.isna().sum().sum())}")
    print(f"{'=' * 60}")
    print(df.dtypes.to_string())
    print()

# ── Speed trace plot ──────────────────────────────────────────────────────────
# Plotting Speed vs Distance gives the classic "speed trace" used by race
# engineers to compare driver braking points and acceleration zones around
# the lap.  Monaco is a street circuit with many slow corners, so expect
# a very spiky trace with low minimum speeds (~60-80 km/h in hairpins).
plt.figure(figsize=(10, 4))
plt.plot(car_df['Distance'], car_df['Speed'])
plt.title(f"Fastest Lap Speed Trace ({fastest['Driver']})")  # fastest['Driver'] = 3-letter code, e.g. 'HAM'
plt.xlabel('Distance (m)')   # position along the lap measured from the start/finish line
plt.ylabel('Speed (km/h)')
plt.tight_layout()
plt.show()

In [ ]:
# ── YOUR TURN (15 min) ────────────────────────────────────────────
# Load the 2023 Italian Grand Prix (Monza) RACE session and answer
# three concrete questions. Follow the steps exactly.
#
# ── Part (a): Who gained the most positions? ──────────────────────
#   1. Load the session:
#        monza = fastf1.get_session(2023, 'Italy', 'R')
#        monza.load(laps=True, telemetry=True, weather=False, messages=False)
#   2. Compute position gain for every driver:
#        monza.results['gain'] = monza.results['GridPosition'] - monza.results['Position']
#      (positive gain = moved forward; largest positive = most positions gained)
#   3. Find the driver with the LARGEST gain:
#        best = monza.results.sort_values('gain', ascending=False).iloc[0]
#   4. Print: driver abbreviation, grid position, finish position, gain.
#
# ── Part (b): Plot that driver's speed trace ──────────────────────
#   1. Get that driver's laps:
#        driver_laps = monza.laps.pick_drivers(best['Abbreviation'])
#   2. Pick their fastest lap:
#        fastest_monza = driver_laps.pick_fastest()
#   3. Get telemetry and add distance:
#        car = fastest_monza.get_car_data().add_distance()
#   4. Plot Speed vs Distance (same style as the Monaco plot above).
#      Title must include the driver's 3-letter code.
#
# ── Part (c): Compare Monza vs. Monaco ────────────────────────────
#   In the Markdown cell below, write 2-4 sentences answering:
#   - Does the Monza trace have higher or lower top speed than Monaco?
#   - Does it have more or fewer braking events per lap?
#   - What does this tell you about circuit type (street vs. power circuit)?
#
# You may use AI tools for syntax help. If you do, start a PROMPTS.md entry.
#
# ✅ Expected output: driver code + gain evidence (printed), speed trace plot.
# ❌ Common mistake: Using 'Monza' instead of 'Italy' as the event name.

# TODO: Write your code here
raise NotImplementedError('Replace this line with your implementation.')

### Part (c): Monza vs. Monaco Speed Trace Comparison

**Your comparison here** (2–4 sentences):

> *How does the Monza speed trace differ from Monaco? What does the circuit layout tell you about the shape of each trace?*

<!-- Your answer: -->

### 📝 Reflection

Answer the following in this cell (2-4 sentences each):

1. **What did you learn from this exercise that surprised you?**
2. **Where could this technique fail or mislead you?**
3. **How would you explain this to a non-technical teammate?**


## 3. Jolpica API — Historical Context

**The idea.** The Jolpica API (successor to the retired Ergast API) exposes historical race results, standings, and circuit data via REST endpoints.

**In F1 terms.** It is your long-horizon archive complementing FastF1's session-level depth. *"Jolpica gives you the big picture. FastF1 gives you the microscope. You need both."*

> **Why it matters:** API-driven pipelines improve provenance and reproducibility.

### 🤔 Before you code: make a decision

Answer these questions in this cell before running any code below.

1. **What question are we trying to answer?**
   > What minimal fields define driver - race prediction records?

2. **What does success look like? How will we know if the result is good?**
   > [Student fills this in]

3. **What could go wrong? Name one failure mode.**
   > [Student fills this in]


In [ ]:
# ── Jolpica API: Historical Race Results ──────────────────────────────────────
# The Jolpica API (api.jolpi.ca) is the community successor to Ergast,
# which was retired at the end of 2024. It's a drop-in replacement using
# the same endpoint structure.

# ── Build the API URL ─────────────────────────────────────────────────────────
# The Jolpica API follows a RESTful pattern:
#   /ergast/f1/{season}/{endpoint}.json
# We request ALL race results for the 2024 season.
# ?limit=1000 overrides the default page size (30) so we get all races in one call
# instead of having to paginate through multiple requests.
results_url = 'https://api.jolpi.ca/ergast/f1/2024/results.json?limit=1000'

# requests.get() sends an HTTP GET request to the URL.
# timeout=30 means: raise an error if the server doesn't respond within 30 seconds.
# .json() parses the JSON response body into a nested Python dict/list structure.
payload = requests.get(results_url, timeout=30).json()

# ── Parse the nested JSON into a flat list of dicts ───────────────────────────
# The API returns a deeply nested structure:
#   payload
#   └── MRData
#       └── RaceTable
#           └── Races  ← list of race objects
#               └── Results  ← list of driver result objects per race
#
# We flatten this into one row per driver per race so it fits in a DataFrame.

rows = []
for race in payload['MRData']['RaceTable']['Races']:
    # Each 'race' dict contains metadata (name, circuit, date) and a Results list
    for result in race.get('Results', []):
        # result['Driver'] and result['Constructor'] are nested dicts
        # We extract only the fields relevant to prediction tasks:
        #   round       : race number in the season (1 = first race, 24 = last)
        #   race        : human-readable GP name (e.g. 'Monaco Grand Prix')
        #   driver      : driver family name (e.g. 'Verstappen')
        #   constructor : team name (e.g. 'Red Bull')
        #   grid        : starting position on the grid (1 = pole position)
        #   position    : finishing position (string — can be 'R' for Retired, 'D' for Disqualified)
        #   points      : championship points awarded for this result
        rows.append(
            {
                'round': int(race['round']),
                'race': race['raceName'],
                'driver': result['Driver']['familyName'],
                'constructor': result['Constructor']['name'],
                'grid': int(result['grid']),
                'position': result.get('position'),   # .get() returns None if key missing (DNF cases)
                'points': float(result.get('points', 0.0)),  # default 0 if no points field present
            }
        )

# Convert the flat list of dicts into a pandas DataFrame.
# Each row = one driver's result at one race.
ergast_2024 = pd.DataFrame(rows)
display(ergast_2024.head())

# ── Fetch Constructor (Team) Championship Standings ───────────────────────────
# This endpoint returns the FINAL standings for the full 2024 season.
# Constructor standings aggregate points across both drivers of each team.
# Useful as a team-strength feature for pre-race predictions.
constructor_url = 'https://api.jolpi.ca/ergast/f1/2024/constructorStandings.json'
constructor_payload = requests.get(constructor_url, timeout=30).json()

# Navigate the nested JSON:
#   StandingsTable → StandingsLists → [0] (only one season) → ConstructorStandings
standings = constructor_payload['MRData']['StandingsTable']['StandingsLists'][0]['ConstructorStandings']

# Build a clean DataFrame with one row per constructor (team).
# position : final championship rank (1 = champion team)
# points   : total points scored across the entire season by both drivers
constructors_df = pd.DataFrame(
    [
        {
            'position': int(x['position']),
            'constructor': x['Constructor']['name'],
            'points': float(x['points']),
        }
        for x in standings
    ]
)
print('\n2024 constructor standings:')
# head(10) shows all 10 teams — F1 always has exactly 10 constructors per season
display(constructors_df.head(10))

In [ ]:
# ── YOUR TURN (10 min) ────────────────────────────────────────────
# Use the Jolpica API to fetch Max Verstappen's championship points
# for THREE seasons (2022, 2023, 2024) and display a comparison table.
#
# Follow these steps exactly:
#
# Step 1. Build a loop over seasons [2022, 2023, 2024].
#         For each season, fetch:
#           url = f'https://api.jolpi.ca/ergast/f1/{season}/driverStandings.json'
#           payload = requests.get(url, timeout=30).json()
#
# Step 2. Navigate the JSON to the standings list:
#           standings_list = payload['MRData']['StandingsTable']['StandingsLists'][0]['DriverStandings']
#
# Step 3. Find Verstappen in the list. Filter by:
#           driver['Driver']['familyName'] == 'Verstappen'
#         Extract: season, position (int), points (float), wins (int).
#
# Step 4. Build a DataFrame with columns: season, position, points, wins.
#         It should have exactly 3 rows (one per season).
#
# Step 5. Print the table AND print one sentence describing the trend:
#         - Did points go up or down? By how much?
#         - Did number of wins change?
#
# ✅ Expected output: 3-row table + 1 trend sentence (printed).
# ❌ Common mistake: The 'wins' field is inside each driver standings entry,
#    not inside the Driver sub-dict — check the JSON structure.

# TODO: Write your code here
raise NotImplementedError('Replace this line with your implementation.')

### 📝 Reflection

Answer the following in this cell (2-4 sentences each):

1. **What did you learn from this exercise that surprised you?**
2. **Where could this technique fail or mislead you?**
3. **How would you explain this to a non-technical teammate?**


## 4. Potential Prediction Targets

**The idea.** Target design should combine business value, feasibility, and availability at prediction time.

**In F1 terms.** Teams only care about predictions that can change strategic decisions.

> **Why it matters:** Bad target definition causes bad metrics and bad models.


### 🤔 Before you code: make a decision

Answer these questions in this cell before running any code below.

1. **What question are we trying to answer?**
   > Which three targets make the strongest case for near-term modeling work and why?

2. **What does success look like? How will we know if the result is good?**
   > [Student fills this in]

3. **What could go wrong? Name one failure mode.**
   > [Student fills this in]


In [ ]:
targets = pd.DataFrame(
    {
        'target': [
            'Race winner (top-1)',
            'Top-10 finish',
            'Pit stop duration',
            'DNF probability',
            'Lap time',
        ],
        'type': [
            'Classification',
            'Binary classification',
            'Regression',
            'Binary classification',
            'Regression',
        ],
        'why_interesting': [
            'High stakes — betting, media, strategy',
            'Business relevant — points threshold',
            'Operational — pit crew and strategy decisions',
            'Safety/risk — reliability planning',
            'Foundational — fine-grained pace signal',
        ],
        'why_hard': [
            'Rare event, small N per season',
            'Imbalanced classes, incident-sensitive',
            'Noisy, many hidden confounders',
            'Very rare, data sparse',
            'Strong temporal dependence',
        ],
    }
)
print(targets.to_string(index=False))

# Discussion question:
# "Before we pick one — what business question would an F1 team actually want answered?"

In [ ]:
# ── YOUR TURN (5 min, in pairs) ───────────────────────────────────
# Propose 2 NEW prediction targets for F1 that are NOT already in the
# `targets` table above. Build a DataFrame and print it.
#
# Rules:
#   - At least ONE target must be regression (not classification).
#   - Each target must answer: "Who would pay for this prediction, and why?"
#
# Follow these steps:
#
# Step 1. Create a DataFrame called `new_targets` with these EXACT columns:
#           target          → name of what you're predicting (e.g. 'Safety car probability')
#           type            → 'Regression' or 'Binary classification'
#           label           → the exact variable being predicted (e.g. 'P(safety_car) per race')
#           prediction_unit → what one row represents (e.g. 'per race', 'per lap', 'per driver')
#           business_case   → one sentence: who benefits and how (e.g. 'Betting markets price SC risk')
#
# Step 2. The DataFrame must have exactly 2 rows.
#
# Step 3. Print it:
#           print(new_targets.to_string(index=False))
#
# Example row (do NOT reuse this one):
#   target='Tire degradation rate', type='Regression',
#   label='seconds lost per lap', prediction_unit='per stint per driver',
#   business_case='Strategists decide when to pit based on projected deg'
#
# ✅ Expected output: 2-row, 5-column table printed to screen.
# ❌ Common mistake: Both targets being classification. One must be regression.

# TODO: Write your code here
raise NotImplementedError('Replace this line with your implementation.')

### 📝 Reflection

Answer the following in this cell (2-4 sentences each):

1. **What did you learn from this exercise that surprised you?**
2. **Where could this technique fail or mislead you?**
3. **How would you explain this to a non-technical teammate?**


## 5. Data Quality First Look

**The idea.** Before modeling, we inspect missingness and structural gaps.

**In F1 terms.** Engineers distrust data feeds until quality checks pass.

> **Why it matters:** Undetected data defects produce misleading confidence.


### 🤔 Before you code: make a decision

Answer these questions in this cell before running any code below.

1. **What question are we trying to answer?**
   > What quality issue is most dangerous for Top-10 prediction: missing grid, wrong label, or stale season mapping?

2. **What does success look like? How will we know if the result is good?**
   > [Student fills this in]

3. **What could go wrong? Name one failure mode.**
   > [Student fills this in]


In [ ]:
nulls = ergast_2024.isna().sum().sort_values(ascending=False)
print('Null counts:')
print(nulls[nulls > 0])

quality_snapshot = {
    'rows': len(ergast_2024),
    'unique_races': int(ergast_2024['round'].nunique()),
    'grid_zero_rows': int((ergast_2024['grid'] == 0).sum()),
}
print('Quality snapshot:')
for k, v in quality_snapshot.items():
    print(f'- {k}: {v}')


In [ ]:
# ── YOUR TURN (10 min) ────────────────────────────────────────────
# Write a reusable pre-modeling quality checklist for `ergast_2024`.
# The checklist must contain AT LEAST 5 automated checks that each
# print PASS or FAIL. This is code you will reuse in Week 2.
#
# Implement these 5 checks (you may add more):
#
# Check 1 — Row count:
#   PASS if ergast_2024 has >= 400 rows (20 drivers × 24 races = 480 expected).
#   Print: f"[{'PASS' if ... else 'FAIL'}] Row count: {len(ergast_2024)}"
#
# Check 2 — No duplicate (round, driver) pairs:
#   PASS if ergast_2024.duplicated(subset=['round', 'driver']).sum() == 0
#
# Check 3 — Grid position range:
#   PASS if grid values are all between 0 and 20 (0 = pit-lane start).
#   Use: ergast_2024['grid'].between(0, 20).all()
#
# Check 4 — Points are non-negative:
#   PASS if (ergast_2024['points'] >= 0).all()
#
# Check 5 — No null values in critical columns:
#   PASS if ergast_2024[['round', 'driver', 'constructor', 'grid']].isna().sum().sum() == 0
#
# ── Output format ─────────────────────────────────────────────────
# Print each check on one line like this:
#   [PASS] Row count: 480
#   [PASS] No duplicate (round, driver) pairs
#   [FAIL] Grid position range — 3 values outside [0, 20]
#
# ✅ Expected output: 5+ lines, each starting with [PASS] or [FAIL].
# ❌ Common mistake: Hardcoding True instead of computing from data.

# TODO: Write your code here
raise NotImplementedError('Replace this line with your implementation.')

In [ ]:
# ── REFERENCE IMPLEMENTATION ──────────────────────────────────────
# This cell is intentionally left sparse in the student version.
# See solutions/ branch for full implementation.


### 📝 Reflection

Answer the following in this cell (2-4 sentences each):

1. **What did you learn from this exercise that surprised you?**
2. **Where could this technique fail or mislead you?**
3. **How would you explain this to a non-technical teammate?**


## 📋 Before Monday, March 9

> **Your next class is Monday. Come with your environment working and Lab 0 in progress.**

- [ ] If you haven't completed the **Pre-Course Diagnostic** — do it today. It gates Lab 0.
- [ ] **Lab 0 is assigned** (see Module 2 on Canvas). Due: **Wednesday March 11, 23:59**. Start now.
  - Your repo needs: `environment.yml`, `setup_check.ipynb`, `data_check.ipynb`, `PROMPTS.md`, `README.md`.
- [ ] Read **Burkov Ch. 1–2** (free PDF — link on Canvas Module 2).
- [ ] Finish your **environment setup**: Python 3.10+, Jupyter, Git, FastF1 installed and working.
- [ ] Continue **GitHub Education** activation if not done yet.
- [ ] Post at least one question or observation in the **Week 1 Discussion Forum** (Module 2).

> *"Lab 0 is the first graded deliverable. It's worth 3% and it's individual. The rubric is on Canvas — read it before you start, not after."*

**Questions?** Post them in the Q&A Forum or come to office hours. Monday we start EDA.

## 🎟️ Exit Ticket — Day 2

Before you leave, answer these 3 questions (submit in Canvas — Exit Ticket Week 1 Wednesday):

1. **Name one thing about the F1 data structure that surprised you.**
2. **Which prediction target from today interests you most, and why?**
3. **On a scale of 1–5, how confident are you that you can complete Lab 0 by March 11?**

> *"This takes 2 minutes. It's not graded, but it tells me where you are — and that information helps me help you. Complete it before you leave."*

## 🤖 PROMPTS.md Workshop — Your First Entry

If you used any AI tool (Copilot, Claude, ChatGPT, etc.) during the Monza task or any part of today's session — you now have material for your first PROMPTS.md entry. If you didn't use AI, use one now: ask it to explain one thing from today's session that confused you. Then document the interaction.

> *"This is not busywork. This is the skill that separates someone who uses AI from someone AI uses. It's also 20% of your lab rubrics — and I read every entry."*

Use the **6-field template** below. Copy this into your `PROMPTS.md` file:

```markdown
## Entry 1 — [Date] — [Brief description]

**Context:** What were you trying to accomplish?
**Prompt(s):** What did you ask the AI? (Paste the actual prompt)
**Output:** What did the AI return? (Summary or key excerpt)
**Validation:** How did you verify the output was correct?
**Adaptations:** What did you change and why?
**Final Decision:** Used / Partially used / Rejected — and why?
```

### Your entry below (fill in all 6 fields):

**Context:**
> [What were you trying to accomplish?]

**Prompt(s):**
> [What did you ask the AI? Paste the actual prompt.]

**Output:**
> [What did the AI return? Summary or key excerpt.]

**Validation:**
> [How did you verify the output was correct?]

**Adaptations:**
> [What did you change and why?]

**Final Decision:**
> [Used / Partially used / Rejected — and why?]